# Football Analysis — Tracking GPU tự động (Colab)

Chỉ chạy **YOLO + BoT-SORT + ReID** trên GPU. Khi bạn upload video trên **web local**, backend tự gửi video sang Colab → tracking → tải stub về → pipeline còn lại chạy trên máy.

## Chuẩn bị (một lần)
1. **Runtime → Change runtime type → T4 GPU**
2. [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) (miễn phí)
3. `best.pt` trên Drive: `MyDrive/football_analysis/best.pt`

## Sau khi chạy cell server
Copy URL ngrok in ra. Trên máy local, tạo file `.env` ở thư mục gốc repo (hoặc set biến môi trường trước khi chạy backend):

```
COLAB_TRACKING_URL=https://xxxx.ngrok-free.app
```

Rồi chạy backend + frontend **local** (không cần `VITE_API_BASE` trỏ Colab):
```bash
uvicorn api.main_api:app --reload --port 8000
cd frontend && npm run dev
```

**Giữ tab Colab mở** trong lúc phân tích. Upload video trên web → Colab tự nhận và tracking.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/PhuongAnh777/football_analysis.git"
BRANCH = "Collab"  # đổi "main" nếu cần

%cd /content

if os.path.isdir("football_analysis/.git"):
    %cd football_analysis
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !rm -rf football_analysis
    !git clone -b {BRANCH} {REPO_URL}
    %cd football_analysis

!git branch --show-current
!ls api/tracking_server.py scripts/run_tracking.py

In [ ]:
# Colab đã có torch+CUDA — không cài lại torch
!pip install -q fastapi "uvicorn[standard]" python-multipart aiofiles requests \
    ultralytics opencv-python numpy pandas scikit-learn scipy matplotlib pyngrok

In [ ]:
import os
from google.colab import drive

DRIVE_MODEL = "/content/drive/MyDrive/football_analysis/best.pt"
MODEL_PATH = "models/best.pt"

os.makedirs("models", exist_ok=True)
os.makedirs("input_videos", exist_ok=True)
os.makedirs("stubs", exist_ok=True)

drive.mount("/content/drive")

if os.path.exists(DRIVE_MODEL):
    !cp "{DRIVE_MODEL}" "{MODEL_PATH}"
    print("Model OK:", MODEL_PATH)
else:
    from google.colab import files
    print("Không thấy model trên Drive. Upload best.pt:")
    uploaded = files.upload()
    !mv best.pt models/best.pt
    print("Model OK:", MODEL_PATH)

In [ ]:
import os
import threading

from pyngrok import ngrok
import uvicorn

# Paste authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""  # ← điền token

if not NGROK_AUTHTOKEN:
    raise ValueError("Hãy điền NGROK_AUTHTOKEN trước khi chạy cell này.")

ngrok.set_auth_token(NGROK_AUTHTOKEN)

PORT = 8001
public_tunnel = ngrok.connect(PORT, bind_tls=True)
public_url = public_tunnel.public_url.rstrip("/")

print("=" * 60)
print("Colab Tracking API:", public_url)
print("Health:", f"{public_url}/api/health")
print()
print("Trên máy local (.env hoặc PowerShell):")
print(f"COLAB_TRACKING_URL={public_url}")
print()
print("Backend local:")
print("  uvicorn api.main_api:app --reload --port 8000")
print("Frontend: cd frontend && npm run dev")
print("Upload video trên web → Colab tự tracking.")
print("Giữ tab Colab mở.")
print("=" * 60)


def _run_server():
    uvicorn.run(
        "api.tracking_server:app",
        host="0.0.0.0",
        port=PORT,
        log_level="info",
    )


server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
print("Tracking server đang chạy...")

In [ ]:
# Kiểm tra server (chạy sau cell server)
import requests

base = public_url
h = requests.get(f"{base}/api/health", headers={"ngrok-skip-browser-warning": "true"}, timeout=30)
print("health:", h.status_code, h.json())
assert h.json().get("model_loaded"), "Thiếu models/best.pt"
assert h.json().get("gpu_available"), "Chưa bật GPU hoặc chưa git pull code mới"
print("OK — sẵn sàng nhận video từ web local")

## (Tùy chọn) Chạy tracking thủ công

Nếu không dùng web, vẫn có thể đặt video trên Drive và chạy cell dưới.

In [ ]:
# Bỏ qua nếu chỉ dùng web → Colab tự nhận video qua API
import os
from google.colab import drive

DRIVE_VIDEO = "/content/drive/MyDrive/football_analysis/input_video.mp4"
VIDEO_PATH = "input_videos/input_video.mp4"

if os.path.exists(DRIVE_VIDEO):
    !cp "{DRIVE_VIDEO}" "{VIDEO_PATH}"
    !python scripts/run_tracking.py "{VIDEO_PATH}"
    print("Stub:", "stubs/track_stubs.pkl")
else:
    print("Không có video trên Drive — dùng upload web hoặc upload file ở đây.")